# 第 13 章 · run 入口脚本：把三层组装起来

**这一章你会得到什么**：看清一个真实入口（`run/mini.py`）如何用工厂函数把 model + environment + agent 拼起来，以及配置是怎么“层层合并”的。**每个 use case = 一个 run 脚本**，这是 mini 的核心设计约定。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/run/mini.py` **L55–105** — `main()`；**L99–102** 是组装三层的三行
- `src/minisweagent/models/__init__.py` **L45–62** — `get_model`
- `src/minisweagent/environments/__init__.py` **L30–33** — `get_environment`
- `src/minisweagent/agents/__init__.py** — `get_agent`（agent 工厂，同样的字符串→类→实例套路）
- `src/minisweagent/utils/serialize.py` **L6–29** — `recursive_merge`（config 层层合并）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [ ]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

In [ ]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 概念：组装只有三行

`mini.py` 的核心其实就是：
```python
model = get_model(config=config.get("model", {}))
env   = get_environment(config.get("environment", {}), default_type="local")
agent = get_agent(model, env, config.get("agent", {}), default_type="interactive")
agent.run(run_task)
```
所有复杂度都在“如何构造 config”，而不是在循环里。

In [ ]:
show_source("src/minisweagent/run/mini.py", 68, 105)

## 实验 1：配置的“层层合并”

`mini.py` 把多个来源的 config 用 `recursive_merge` 合并，后者覆盖前者。亲手验证一次。

In [ ]:
from minisweagent.utils.serialize import recursive_merge, UNSET
base = {"agent": {"cost_limit": 3.0, "step_limit": 0}, "model": {"model_name": "gpt-4o"}}
cli  = {"agent": {"cost_limit": 10.0}, "model": {"model_name": UNSET}}  # UNSET 会被跳过
merged = recursive_merge(base, cli)
print("合并后 agent:", merged["agent"])
print("合并后 model:", merged["model"], "  # UNSET 不覆盖，保留原值")

## 实验 2：亲手走一遍 mini.py 的组装（用确定性模型离线跑）

In [ ]:
from minisweagent.models import get_model_class
from minisweagent.environments import get_environment
from minisweagent.agents.default import DefaultAgent
from minisweagent.models.test_models import DeterministicToolcallModel, make_toolcall_output

print("get_model_class('litellm') ->", get_model_class("litellm").__name__)

# 真实里是 get_model(...) 拿 LitellmModel；离线用确定性模型替身
submit = {"command": "echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT\necho assembled", "tool_call_id": "c1"}
model = DeterministicToolcallModel(outputs=[make_toolcall_output("提交", [], [submit])])
env = get_environment({"cwd": str(REPO)}, default_type="local")   # 工厂造 LocalEnvironment
agent = DefaultAgent(model, env, system_template="s", instance_template="{{task}}", cost_limit=5)
print("组装类型:", type(model).__name__, "+", type(env).__name__, "+", type(agent).__name__)
print("run 结果:", agent.run("演示 mini.py 的组装"))

## 观察点
- `get_model` / `get_environment` / `get_agent` 都是“字符串 -> 类 -> 实例”的工厂（回忆第 12 章的环境工厂）。切换实现只改配置里的 `*_class`，不改脚本骨架。
- `mini.py` 默认 agent 是 `interactive`（人在环）；批量评测脚本会换成别的。**同样三层，不同 use case 用不同 run 脚本组装**。
- `UNSET` 这个哨兵值让“命令行没传的选项”不会覆盖配置文件里的值——一个很实用的合并技巧。

## 动手：把 default_type 改成造一个不同环境
用 `get_environment({}, default_type="local")` 和显式 `{"environment_class": "local"}` 两种方式各造一个环境，确认拿到的是同一个类。

In [ ]:
from minisweagent.environments import get_environment
e1 = get_environment({}, default_type="local")
e2 = get_environment({"environment_class": "local"})
print(type(e1).__name__, type(e2).__name__)
# TODO: assert 两者类型相同

## 闭卷检查
1. mini.py 组装三层的三行代码是什么？
2. `recursive_merge` + `UNSET` 解决了什么问题？
3. 为什么“一个 use case 一个 run 脚本”是好设计？